# scOPE end-to-end -- SKCM (TCGA Skin Cutaneous Melanoma)

Two-phase scOPE workflow applied to melanoma:

1. **Phase 1** -- learn a latent space from TCGA SKCM bulk RNA-seq (~470
   samples) and train mutation classifiers (BRAF, NRAS, NF1, TP53, ...).
2. **Phase 2** -- project Jerby-Arnon et al. 2018 melanoma scRNA-seq
   (GSE115978, 7 186 cells, 31 tumours) into the bulk latent space and infer
   per-cell mutation probabilities.

**Data notes:**
- Bulk expression: UCSC Xena TCGA SKCM `HiSeqV2_PANCAN`
  (log2(norm_count + 1), genes x samples).  Both primary (01) and metastatic
  (06) samples are retained.
- Mutations: TCGA MC3 PanCanAtlas MAF (shared with CRC/BRCA notebooks).
  Filtered to SKCM samples by matching against bulk obs_names.
- scRNA-seq: Jerby-Arnon et al. 2018 (GSE115978) -- TPM matrix (log2(TPM+1))
  downloaded via GEO HTTPS endpoint. No further normalisation applied.

**BRAF V600E note:**
BRAF V600E is a hotspot detectable at the RNA level (allele-specific
expression). After projection, you can optionally validate scOPE's predicted
BRAF probability against SComatic or RESA calls from the scRNA-seq BAMs.


## 1. Imports & paths

In [ ]:
import os

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import requests
from sklearn.isotonic import IsotonicRegression

from scope import BulkPipeline, SingleCellPipeline
from scope.visualization import (
    compute_umap,
    plot_mutation_probabilities,
    plot_scree,
    plot_mutation_heatmap,
)


In [ ]:
# -- Paths ------------------------------------------------------------------
BASE_DIR   = "/Users/ashforda/Desktop/Pathways + Omics/scOPE/scOPE_project_overhaul/scOPE/data"
BULK_DIR   = os.path.join(BASE_DIR, "TCGA_SKCM")
SC_DIR     = os.path.join(BASE_DIR, "SKCM_scRNA")
MODELS_DIR = os.path.join(BASE_DIR, "..", "models", "SKCM")

for d in [BULK_DIR, SC_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

# -- File locations ---------------------------------------------------------
skcm_expr_path = os.path.join(BULK_DIR, "TCGA_SKCM_HiSeqV2_PANCAN.tsv.gz")
mc3_path       = os.path.join(BASE_DIR, "mc3.v0.2.8.PUBLIC.xena.gz")
sc_tpm_path    = os.path.join(SC_DIR,   "GSE115978_tpm.csv.gz")
# GEO uses dot separator: GSE115978_cell.annotations.csv.gz
sc_meta_path   = os.path.join(SC_DIR,   "GSE115978_cell.annotations.csv.gz")


## 2. Download raw data

In [ ]:
# -- UCSC Xena -- bulk RNA-seq (expression files still work) ---------------
XENA_BASE = "https://tcga.xenahubs.net/download"

if os.path.exists(skcm_expr_path):
    print(f"  already present -- {os.path.basename(skcm_expr_path)}")
else:
    print(f"  downloading {os.path.basename(skcm_expr_path)} ...")
    r = requests.get(f"{XENA_BASE}/TCGA.SKCM.sampleMap/HiSeqV2_PANCAN.gz",
                     stream=True, timeout=300)
    r.raise_for_status()
    with open(skcm_expr_path, "wb") as fh:
        for chunk in r.iter_content(1 << 20):
            fh.write(chunk)
    print(f"  done -> {skcm_expr_path}")


In [ ]:
# -- MC3 PanCanAtlas MAF -- all TCGA cancer types, one file (~200 MB) -------
# Shared resource -- check CRC and BRCA directories before re-downloading.
for _alt in [
    os.path.join(BASE_DIR, "TCGA_CRC",  "mc3.v0.2.8.PUBLIC.xena.gz"),
    os.path.join(BASE_DIR, "TCGA_BRCA", "mc3.v0.2.8.PUBLIC.xena.gz"),
]:
    if not os.path.exists(mc3_path) and os.path.exists(_alt):
        import shutil
        shutil.copy(_alt, mc3_path)
        print(f"  copied MC3 from {os.path.dirname(_alt)} -> {mc3_path}")
        break

if not os.path.exists(mc3_path):
    print("Downloading MC3 MAF (~200 MB) ...")
    r = requests.get(
        "https://pancanatlas.xenahubs.net/download/mc3.v0.2.8.PUBLIC.xena.gz",
        stream=True, timeout=600,
    )
    r.raise_for_status()
    with open(mc3_path, "wb") as fh:
        for chunk in r.iter_content(1 << 20):
            fh.write(chunk)
    print(f"  done -> {mc3_path}")
else:
    print(f"  already present -- {os.path.basename(mc3_path)}")


In [ ]:
# -- GEO -- Jerby-Arnon et al. 2018 GSE115978 ------------------------------
# TPM matrix  : genes x cells, log2(TPM+1)
# Cell metadata: cell type, tumour, malignancy label
# Note: GEO filename is GSE115978_cell.annotations.csv.gz (dot separator, csv)
GEO_HTTP = "https://www.ncbi.nlm.nih.gov/geo/download"

sc_downloads = {
    sc_tpm_path  : (f"{GEO_HTTP}/?acc=GSE115978&format=file"
                    "&file=GSE115978%5Ftpm%2Ecsv%2Egz"),
    sc_meta_path : (f"{GEO_HTTP}/?acc=GSE115978&format=file"
                    "&file=GSE115978%5Fcell%2Eannotations%2Ecsv%2Egz"),
}
for dest, url in sc_downloads.items():
    if os.path.exists(dest):
        print(f"  already present -- {os.path.basename(dest)}")
    else:
        print(f"  downloading {os.path.basename(dest)} ...")
        r = requests.get(url, stream=True, timeout=600)
        r.raise_for_status()
        with open(dest, "wb") as fh:
            for chunk in r.iter_content(1 << 20):
                fh.write(chunk)
        print(f"  done -> {dest}")


## 3. Load & prepare bulk RNA-seq

SKCM includes both primary (01) and metastatic (06) tumours; both are retained
since the scRNA-seq cohort is also mixed.


In [ ]:
# -- Peek -------------------------------------------------------------------
peek = pd.read_csv(skcm_expr_path, sep="\t", index_col=0, nrows=3)
print(f"Peek : {peek.shape}   cols[:4] = {peek.columns[:4].tolist()}")


In [ ]:
df_bulk = pd.read_csv(skcm_expr_path, sep="\t", index_col=0).T   # samples x genes

# TCGA sample type codes:
#   01 = primary tumour, 06 = metastatic, 11 = normal solid tissue
# Including normals gives the classifier true negatives with distinct
# transcriptional profiles -- this pulls the TME probability floor down.
SAMPLE_TYPE_MAP = {"01": "primary", "06": "metastatic", "11": "normal"}
df_bulk = df_bulk[df_bulk.index.str[13:15].isin(SAMPLE_TYPE_MAP)]

adata_bulk = ad.AnnData(
    X   = df_bulk.values.astype(np.float32),
    obs = pd.DataFrame(
            {"sample_type": df_bulk.index.str[13:15].map(SAMPLE_TYPE_MAP)},
            index=df_bulk.index,
          ),
    var = pd.DataFrame(index=df_bulk.columns),
)
adata_bulk.var_names_make_unique()
print(f"Bulk loaded : {adata_bulk.n_obs} samples x {adata_bulk.n_vars} genes")
print(adata_bulk.obs["sample_type"].value_counts())


In [ ]:
# -- Sanity check -----------------------------------------------------------
X = adata_bulk.X
print(f"NaN : {np.isnan(X).sum()} | Inf : {np.isinf(X).sum()}")
print(f"Min : {np.nanmin(X):.3f} | Max : {np.nanmax(X):.3f}")
print(f"Neg : {(X < 0).sum()}  (expected for log2 data)")


## 4. Build mutation label matrix

SKCM has very high TMB (UV signature). We focus on major driver genes where
expression-linked signatures are strong (BRAF, NRAS, NF1, TP53, CDKN2A).
SKCM samples are identified by matching against `adata_bulk.obs_names`.


In [ ]:
# -- Key SKCM driver genes --------------------------------------------------
SKCM_GENES = [
    # -- MAPK pathway -------------------------------------------------------
    "BRAF",     # ~50% -- V600E hotspot; defines targeted-therapy eligibility
    "NRAS",     # ~30% -- Q61 hotspot; mutually exclusive with BRAF
    "NF1",      # ~15% -- RASopathy gene; third major melanoma driver
    "MAP2K1",   # MEK1
    "MAP2K2",   # MEK2
    "KRAS",
    "HRAS",

    # -- PI3K/AKT pathway ---------------------------------------------------
    "PTEN",     # loss common in BRAF-mutant tumours
    "PIK3CA",
    "AKT1",
    "AKT3",

    # -- Tumour suppressors -------------------------------------------------
    "TP53",
    "CDKN2A",   # p16/ARF locus -- frequent deletion

    # -- WNT / differentiation ----------------------------------------------
    "CTNNB1",
    "APC",

    # -- Chromatin / epigenetic ---------------------------------------------
    "ARID2",
    "SETD2",
    "KDM6A",
    "ARID1A",

    # -- Other recurrent ----------------------------------------------------
    "RAC1",     # P29S -- UV hotspot
    "PPP6C",    # UV hotspot -- MAPK pathway regulation
    "PREX2",    # PI3K pathway
    "IDH1",
    "KIT",
    "GNAQ",     # uveal melanoma
    "GNA11",    # uveal melanoma
]
SKCM_GENES = list(dict.fromkeys(SKCM_GENES))
print(f"Targeting {len(SKCM_GENES)} SKCM driver genes")


In [ ]:
KEEP_CLASSES = {
    "Missense_Mutation", "Nonsense_Mutation", "Frame_Shift_Del",
    "Frame_Shift_Ins", "In_Frame_Del", "In_Frame_Ins",
    "Splice_Site", "Translation_Start_Site", "Nonstop_Mutation",
}


In [ ]:
# -- Load MC3, filter to SKCM by matching bulk obs_names -------------------
mc3 = pd.read_csv(mc3_path, sep="\t", low_memory=False)
print(f"MC3 raw : {len(mc3):,} rows   cols[:6]: {mc3.columns.tolist()[:6]}")

mc3 = mc3.rename(columns={
    "sample" : "Tumor_Sample_Barcode",
    "effect" : "Variant_Classification",
    "gene"   : "Hugo_Symbol",
})

# Filter to SKCM by matching against bulk 15-char barcodes
bulk_15_set = set(adata_bulk.obs_names.str[:15])
mc3["sample_id"] = mc3["Tumor_Sample_Barcode"].str[:15]
maf_skcm = mc3[mc3["sample_id"].isin(bulk_15_set)].copy()
print(f"SKCM rows (pre-filter) : {len(maf_skcm):,}   samples: {maf_skcm['sample_id'].nunique()}")

maf_skcm = maf_skcm[maf_skcm["Variant_Classification"].isin(KEEP_CLASSES)]
maf_all  = maf_skcm[["sample_id", "Hugo_Symbol"]].dropna()
print(f"After coding filter    : {len(maf_all):,} variants   {maf_all['sample_id'].nunique()} samples")


In [ ]:
# -- Build binary mutation matrix (samples x genes) ------------------------
mut_matrix = (
    maf_all[["sample_id", "Hugo_Symbol"]]
    .drop_duplicates()
    .assign(mutated=1)
    .pivot_table(index="sample_id", columns="Hugo_Symbol",
                 values="mutated", fill_value=0)
)
mut_matrix.columns.name = None
mut_matrix.index.name   = None

genes_present = [g for g in SKCM_GENES if g in mut_matrix.columns]
genes_missing = [g for g in SKCM_GENES if g not in mut_matrix.columns]
mutation_labels = mut_matrix[genes_present]

print(f"Mutation matrix : {mutation_labels.shape}")
print(f"Genes found     : {genes_present}")
if genes_missing:
    print(f"Genes missing   : {genes_missing}")


In [ ]:
# -- Intersect tumour samples with MC3; zero-fill normals -----------------
# Normal samples (type 11) have no somatic mutations in MC3 by design.
# We keep them and assign all-zero mutation rows rather than dropping them,
# so the classifier sees genuine mutation-negative transcriptomes.
bulk_15   = pd.Index(adata_bulk.obs_names.str[:15])
mut_15    = pd.Index(mutation_labels.index.str[:15])
is_normal = adata_bulk.obs["sample_type"] == "normal"

has_mc3   = bulk_15.isin(mut_15)
keep_mask = has_mc3 | is_normal.values
print(f"Tumour samples with MC3 : {has_mc3.sum()}")
print(f"Normal samples kept     : {is_normal.sum()}")
print(f"Total retained          : {keep_mask.sum()} / {adata_bulk.n_obs}")
assert has_mc3.sum() > 50, "Suspiciously few MC3 matches -- check barcode format"

adata_bulk = adata_bulk[keep_mask].copy()
adata_bulk.obs["barcode_15"] = adata_bulk.obs_names.str[:15]

# Align mutation labels: tumour rows from MC3, normal rows = 0
mut_reindexed       = mutation_labels.copy()
mut_reindexed.index = mut_reindexed.index.str[:15]
mut_reindexed       = mut_reindexed[~mut_reindexed.index.duplicated(keep="first")]

# Reindex -- normals get NaN then fill 0
mutation_labels = mut_reindexed.reindex(adata_bulk.obs["barcode_15"]).fillna(0).astype(int)
mutation_labels.index = adata_bulk.obs_names

print(f"\nMutation matrix : {mutation_labels.shape}")
print("\nMutation frequencies (tumour samples only):")
tumour_mask = adata_bulk.obs["sample_type"] != "normal"
print(mutation_labels[tumour_mask].sum().sort_values(ascending=False))
mutation_labels.head()


## 5. Load single-cell data

In [ ]:
# -- Jerby-Arnon et al. 2018 GSE115978 -- log2(TPM+1) ----------------------
# File is genes x cells, already log2-scaled -- no CPM normalisation needed.
print("Loading TPM matrix ...")
tpm_df = pd.read_csv(sc_tpm_path, index_col=0)   # genes x cells
print(f"TPM raw : {tpm_df.shape}  (genes x cells)")


In [ ]:
# -- Cell metadata ---------------------------------------------------------
# GSE115978_cell.annotations.csv.gz is comma-separated, not TSV.
meta = pd.read_csv(sc_meta_path, index_col=0)
print(f"Metadata shape   : {meta.shape}")
print(f"Metadata columns : {meta.columns.tolist()}")
meta.head(3)


In [ ]:
# -- Build AnnData ---------------------------------------------------------
common_cells = tpm_df.columns.intersection(meta.index)
print(f"Cells in both matrix and metadata : {len(common_cells)}")

adata_sc = ad.AnnData(
    X   = tpm_df[common_cells].T.values.astype(np.float32),
    obs = meta.loc[common_cells],
    var = pd.DataFrame(index=tpm_df.index),
)
adata_sc.var_names_make_unique()
print(f"\nSC loaded : {adata_sc.n_obs} cells x {adata_sc.n_vars} genes")

ct_col = next(
    (c for c in adata_sc.obs.columns
     if "type" in c.lower() or "class" in c.lower()), None
)
if ct_col:
    print(f"\nCell type distribution ({ct_col}):")
    print(adata_sc.obs[ct_col].value_counts())


In [ ]:
# -- QC (TPM already log2-scaled -- skip CPM normalisation) ----------------
sc.pp.filter_genes(adata_sc, min_cells=5)
sc.pp.filter_cells(adata_sc, min_genes=100)

print(f"After QC : {adata_sc.n_obs} cells x {adata_sc.n_vars} genes")

X_raw = adata_sc.X
print(f"Value range: {X_raw.min():.2f} - {X_raw.max():.2f}")
print("(GSE115978 TPM matrix is log2(TPM+1) -- no further normalisation applied.)")


## 6. Gene overlap & sanity checks

In [ ]:
overlap = set(adata_bulk.var_names) & set(adata_sc.var_names)
print(f"Bulk  : {adata_bulk.n_obs} samples x {adata_bulk.n_vars} genes")
print(f"SC    : {adata_sc.n_obs} cells x {adata_sc.n_vars} genes")
print(f"Muts  : {mutation_labels.shape}")
print(f"Gene overlap : {len(overlap):,}")

X = adata_bulk.X
print(f"\nBulk NaN : {np.isnan(X).sum()} | Inf : {np.isinf(X).sum()}")
print(f"\nMutation frequencies:")
print(mutation_labels.sum().sort_values(ascending=False))
mutation_labels.head()


## 7. Phase 1 -- Bulk pipeline

Both bulk and SC data are log2-scaled, so `norm_method='none'` and
`log1p=False` are appropriate for both phases.


In [ ]:
n_components = 100

bulk_pipe = BulkPipeline(
    norm_method         = "none",
    log1p               = False,
    center              = True,
    scale               = True,
    decomposition       = "svd",
    n_components        = n_components,
    classifier          = "logistic",
    min_positive_frac   = 0.0001,
    classifier_kwargs   = {
        "C"         : 1.0,
        "solver"    : "lbfgs",
        "max_iter"  : 100000,
    },
)

bulk_pipe.fit(adata_bulk, mutation_labels, cv=5)
bulk_pipe.save(os.path.join(MODELS_DIR, "SKCM_bulk_pipeline.pkl"))
print("Phase 1 complete -- model saved.")


In [ ]:
scree = bulk_pipe.decomposer_.scree_data()
fig, ax = plot_scree(scree, max_components=n_components)
plt.tight_layout()
fig.savefig(os.path.join(MODELS_DIR, "SKCM_scree.pdf"), bbox_inches="tight")
plt.show()


## 8. Phase 2 -- Single-cell pipeline

In [ ]:
adata_bulk_pp = bulk_pipe.preprocessor_.transform(adata_bulk)

sc_pipe = SingleCellPipeline(
    bulk_pipeline    = bulk_pipe,
    alignment_method = "moment_matching",
)
sc_pipe.fit(adata_bulk_pp, adata_sc)
adata_sc = sc_pipe.transform(adata_sc)

mut_prob_cols = [c for c in adata_sc.obs.columns if c.startswith("mutation_prob_")]
print(f"Inferred mutation probability columns ({len(mut_prob_cols)}): {mut_prob_cols}")

# Store raw (pre-calibration) probabilities for comparison
for col in mut_prob_cols:
    adata_sc.obs[col.replace("mutation_prob_", "mutation_prob_raw_")] = adata_sc.obs[col].values


## 9. Post-hoc probability calibration

The logistic classifier outputs probabilities relative to the bulk training
distribution. With ~50% BRAF-mutant samples, the prior is high and TME cells
inherit an elevated floor. We apply **isotonic regression calibration** fitted
on bulk leave-one-out predictions vs. known mutation labels, then re-apply to
the SC probabilities. This preserves rank order while correcting the baseline.


In [ ]:
# -- Isotonic calibration on bulk CV predictions --------------------------
# Get bulk predicted probabilities from the fitted pipeline
bulk_probs_raw = bulk_pipe.predict_proba(adata_bulk)   # shape: (n_bulk, n_genes)

genes_for_calib = mutation_labels.columns.tolist()
calibrators     = {}

for gene in genes_for_calib:
    y_true = mutation_labels[gene].values
    # Only calibrate genes with enough positives
    if y_true.sum() < 10:
        continue
    gene_idx  = genes_for_calib.index(gene)
    y_score   = bulk_probs_raw[:, gene_idx]
    iso       = IsotonicRegression(out_of_bounds="clip")
    iso.fit(y_score, y_true)
    calibrators[gene] = iso

print(f"Fitted calibrators for {len(calibrators)} genes")

# Apply calibration to SC probabilities
for gene, iso in calibrators.items():
    raw_col  = f"mutation_prob_{gene}"
    calib_col = f"mutation_prob_cal_{gene}"
    if raw_col in adata_sc.obs.columns:
        adata_sc.obs[calib_col] = iso.predict(adata_sc.obs[raw_col].values)

calib_cols = [c for c in adata_sc.obs.columns if c.startswith("mutation_prob_cal_")]
print(f"Calibrated probability columns: {calib_cols}")


In [ ]:
# -- Diagnostic: P(BRAF) distribution by cell type, raw vs calibrated -----
ct_col = next((c for c in adata_sc.obs.columns
               if "type" in c.lower() or "class" in c.lower()), None)

if ct_col and "mutation_prob_cal_BRAF" in adata_sc.obs.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=False)
    cell_types = adata_sc.obs[ct_col].unique()

    for ax, col, title in zip(
        axes,
        ["mutation_prob_BRAF", "mutation_prob_cal_BRAF"],
        ["P(BRAF) -- raw", "P(BRAF) -- isotonic calibrated"],
    ):
        data   = [adata_sc.obs.loc[adata_sc.obs[ct_col] == ct, col].values
                  for ct in cell_types]
        vp = ax.violinplot(data, showmedians=True)
        ax.set_xticks(range(1, len(cell_types) + 1))
        ax.set_xticklabels(cell_types, rotation=45, ha="right")
        ax.set_ylabel("Probability")
        ax.set_title(title)
        ax.axhline(0.5, color="grey", lw=0.8, linestyle="--", label="0.5")
    plt.tight_layout()
    fig.savefig(os.path.join(MODELS_DIR, "SKCM_BRAF_calibration_diagnostic.pdf"),
                bbox_inches="tight")
    plt.show()
else:
    print("ct_col or calibrated BRAF column not found -- skipping diagnostic.")


## 10. Visualisation


In [ ]:
adata_sc = compute_umap(adata_sc, obsm_key="X_svd")

ct_col = next((c for c in adata_sc.obs.columns
               if "type" in c.lower() or "class" in c.lower()), "sample_type")

# Prefer calibrated BRAF prob if available
braf_col = ("mutation_prob_cal_BRAF"
            if "mutation_prob_cal_BRAF" in adata_sc.obs.columns
            else "mutation_prob_BRAF")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.umap(adata_sc, color=ct_col,   ax=axes[0], show=False, title="Cell type")
sc.pl.umap(adata_sc, color=braf_col, ax=axes[1], show=False,
           title=f"P(BRAF mutated) [{braf_col}]")
plt.tight_layout()
fig.savefig(os.path.join(MODELS_DIR, "SKCM_umap_BRAF.pdf"), bbox_inches="tight")
plt.show()


In [ ]:
top_muts = (
    mutation_labels.sum().sort_values(ascending=False).head(6).index.tolist()
)
# Use calibrated columns where available, fall back to raw
def best_prob_col(gene):
    cal = f"mutation_prob_cal_{gene}"
    raw = f"mutation_prob_{gene}"
    return cal if cal in adata_sc.obs.columns else raw

fig = plot_mutation_probabilities(adata_sc, mutations=top_muts,
                                   prob_col_fn=best_prob_col)
fig.savefig(os.path.join(MODELS_DIR, "SKCM_mutation_probs.pdf"), bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plot_mutation_heatmap(adata_sc, cluster_key=ct_col, mutations=top_muts)
fig.savefig(os.path.join(MODELS_DIR, "SKCM_heatmap.pdf"), bbox_inches="tight")
plt.show()
